## Building A Chatbot
An example of how to design and implement an LLM-powered chatbot. This chatbot will be able to have a conversation and remember previous interactions.

Note that this chatbot that we build will only use the language model to have a conversation. There are several other related concepts that we may be looking for:

- Conversational RAG: Enable a chatbot experience over an external source of data
- Agents: Build a chatbot that can take actions

This example will cover the basics which will be helpful for those two more advanced topics.

In [1]:
import os
from dotenv import load_dotenv

load_dotenv("../../.env")

groq_api_key = os.getenv("GROQ_API_KEY")

In [2]:
from langchain_groq import ChatGroq

model = ChatGroq(model="llama-3.1-8b-instant", groq_api_key=groq_api_key) #llama-3.1-8b-instant
model
 

e:\GitHub\Python\Python_Agentic_AI_With_Langchain_and_Langraph\practice_agentic_ai\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 8192, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x0000028455F9FA10>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x00000284561A4830>, model_name='llama-3.1-8b-instant', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [3]:
from langchain_core.messages import HumanMessage

model.invoke([HumanMessage(content="Hi, my name is Vaibhav, and I am a software engineer.")])


AIMessage(content="Nice to meet you, Vaibhav. I'm happy to chat with you about software engineering, technology, or anything else you'd like to discuss. What brings you here today? Are you working on a specific project, or do you have some questions about a particular topic in software engineering?", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 61, 'prompt_tokens': 52, 'total_tokens': 113, 'completion_time': 0.162623126, 'completion_tokens_details': None, 'prompt_time': 0.00259556, 'prompt_tokens_details': None, 'queue_time': 0.052129649, 'total_time': 0.165218686}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019d8c77-386e-7c30-9814-1cda2357552e-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 52, 'output_tokens': 61, 'total_tokens': 113})

In [4]:
from langchain_core.messages import AIMessage

model.invoke([
    HumanMessage(content="Hi, my name is Vaibhav, and I am a software engineer."),
    AIMessage(content="Hello Vaibhav! It's great to meet you. How can I assist you today?"),
    HumanMessage(content="Hey, what is my name and what do I do?")
])

AIMessage(content='Your name is Vaibhav, and you are a software engineer.', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 16, 'prompt_tokens': 94, 'total_tokens': 110, 'completion_time': 0.024063818, 'completion_tokens_details': None, 'prompt_time': 0.007773291, 'prompt_tokens_details': None, 'queue_time': 0.053233343, 'total_time': 0.031837109}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019d8c77-3bd9-7b12-9d3d-29d1bdd77168-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 94, 'output_tokens': 16, 'total_tokens': 110})

### Message History
We can use a Message History class to wrap our model and make it stateful. This will keep track of inputs and outputs of the model, and store them in some datastore. Future interactions will then load those messages and pass them into the chain as part of the input. Let's see how to use this!

In [5]:
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.runnables import RunnableWithMessageHistory

store = {}

def get_session_history(session_id: str) -> BaseChatMessageHistory:
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]

with_message_history = RunnableWithMessageHistory(model, get_session_history)

In [6]:
config = {"configurable":
            {"session_id": "chat1"}
    }

In [7]:
response = with_message_history.invoke(
    [HumanMessage(content="Hi, my name is Vaibhav, and I am a software engineer.")],
    config=config
)   

In [8]:
response.content

"Nice to meet you, Vaibhav. I'm happy to chat with you about software engineering or any other topic you'd like to discuss. What brings you here today? Are you working on a specific project or looking for advice on a particular issue?"

In [9]:
response = with_message_history.invoke(
    [HumanMessage(content="What is my name")],
    config=config
)
response.content

'Your name is Vaibhav.'

In [10]:
# Change the config =>>>>> change the config

config1 = {"configurable":
            {"session_id": "chat2"}
        }
response = with_message_history.invoke(
    [HumanMessage(content="What is my name?")]
    ,config=config1
)
response.content

"I don't have any information about your name. I'm a large language model, I don't have the ability to retain information about individual users or their personal details. Each time you interact with me, it's a new conversation and I don't have any prior knowledge about you. If you'd like to share your name with me, I'd be happy to chat with you!"

### Prompt templates
Prompt Templates help to turn raw user information into a format that the LLM can work with. In this case, the raw user input is just a message, which we are passing to the LLM. Let's now make that a bit more complicated. First, let's add in a system message with some custom instructions (but still taking messages as input). Next, we'll add in more input besides just the messages.

In [11]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

prompt = ChatPromptTemplate.from_messages(
    [       
        ("system", "You are a helpful assistant. Answer the question based on the conversation history."), 
        MessagesPlaceholder(variable_name="messages")
    ]
)

chain = prompt | model

In [12]:
chain.invoke(
    {"messages": [HumanMessage(content="Hi, my name is Vaibhav, and I am a software engineer.")]}
)

AIMessage(content="Nice to meet you, Vaibhav. It's great to hear that you're a software engineer. What brings you here today? Do you have any specific questions or topics you'd like to discuss related to software development?", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 47, 'prompt_tokens': 67, 'total_tokens': 114, 'completion_time': 0.063789187, 'completion_tokens_details': None, 'prompt_time': 0.004391274, 'prompt_tokens_details': None, 'queue_time': 0.16008787, 'total_time': 0.068180461}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_7ccc667439', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019d8c77-41d5-7842-b2dc-8641492f3a7d-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 67, 'output_tokens': 47, 'total_tokens': 114})

In [13]:
# invoke with chat message history

with_message_history=RunnableWithMessageHistory(chain, get_session_history)

In [14]:
config = {"configurable": {"session_id": "chat3"}}
response = with_message_history.invoke(
    {
        "messages": [HumanMessage(content="Hi, my name is Vaibhav, and I am a software engineer.")]},
        config=config
)
response.content

"Hello Vaibhav, nice to meet you. It's great to hear that you're a software engineer. What brings you here today? Are you working on a new project or do you have any questions or topics you'd like to discuss?"

In [15]:
# Add more complexity
prompt = ChatPromptTemplate.from_messages(
    [       
        (
            "system", "You are a helpful assistant. Answer all the question to the best of your ability in {language} language."
        ), 
        MessagesPlaceholder(variable_name="messages")
    ]
)
chain = prompt | model

In [16]:
chain.invoke(
    {
        "messages": [HumanMessage(content="Hi, my name is Vaibhav.")],
        "language": "Hindi"
    }
)
response.content

"Hello Vaibhav, nice to meet you. It's great to hear that you're a software engineer. What brings you here today? Are you working on a new project or do you have any questions or topics you'd like to discuss?"

Let's now wrap this more complicated chain in a Message History class. This time, because there are multiple keys in the input, we need to specify the correct key to use to save the chat history.

In [17]:
with_message_history=RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="messages"
)

In [18]:
config = {"configurable":
            {"session_id": "chat4"}
        }
response = with_message_history.invoke(
    {
        "messages": [HumanMessage(content="Hi, my name is Vaibhav.")],
        "language": "Hindi"
    },
    config=config
)
response.content

'नमस्ते वैभव जी! मैं आपकी मदद करने के लिए यहाँ हूँ। क्या आपके पास कोई प्रश्न या समस्या है जिससे मैं आपकी सहायता कर सकता हूँ?'

### Managing the Conversation History
One important concept to understand when building chatbots is how to manage conversation history. If left unmanaged, the list of messages will grow unbounded and potentially overflow the context window of the LLM. Therefore, it is important to add a step that limits the size of the messages you are passing in.

- 'trim_messages' helper to reduce how many messages we're sending to the model. The trimmer allows us to specify how many tokens we want to keep, along with other parameters like if we want to always keep the system message and whether to allow partial messages

In [21]:
from langchain_core.messages import SystemMessage, trim_messages
from langchain_core.messages import HumanMessage, AIMessage, HumanMessage

trimmer=trim_messages(
    max_tokens=45,
    strategy="last",
    token_counter=model,
    include_system=True,
    allow_partial=False,
    start_on="human"
)
messages = [
    SystemMessage(content="you're a good assistant"),
    HumanMessage(content="hi! I'm bob"),
    AIMessage(content="hi!"),
    HumanMessage(content="I like vanilla ice cream"),
    AIMessage(content="nice"),
    HumanMessage(content="whats 2 + 2"),
    AIMessage(content="4"),
    HumanMessage(content="thanks"),
    AIMessage(content="no problem!"),
    HumanMessage(content="having fun?"),
    AIMessage(content="yes!"),
]
trimmer.invoke(messages)

HTTP Error 503 thrown while requesting HEAD https://huggingface.co/gpt2/resolve/main/merges.txt
Retrying in 1s [Retry 1/5].
HTTP Error 503 thrown while requesting HEAD https://huggingface.co/gpt2/resolve/main/merges.txt
Retrying in 2s [Retry 2/5].
HTTP Error 503 thrown while requesting HEAD https://huggingface.co/gpt2/resolve/main/merges.txt
Retrying in 4s [Retry 3/5].
HTTP Error 503 thrown while requesting HEAD https://huggingface.co/gpt2/resolve/main/merges.txt
Retrying in 8s [Retry 4/5].
HTTP Error 504 thrown while requesting HEAD https://huggingface.co/gpt2/resolve/main/merges.txt
Retrying in 8s [Retry 5/5].
HTTP Error 504 thrown while requesting HEAD https://huggingface.co/gpt2/resolve/main/merges.txt


[SystemMessage(content="you're a good assistant", additional_kwargs={}, response_metadata={}),
 HumanMessage(content='I like vanilla ice cream', additional_kwargs={}, response_metadata={}),
 AIMessage(content='nice', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='whats 2 + 2', additional_kwargs={}, response_metadata={}),
 AIMessage(content='4', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='thanks', additional_kwargs={}, response_metadata={}),
 AIMessage(content='no problem!', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='having fun?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='yes!', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]

In [25]:
from operator import itemgetter
from langchain_core.runnables import RunnablePassthrough

chain = (
    RunnablePassthrough.assign(messages=itemgetter("messages") | trimmer )
    | prompt
    | model
)
response=chain.invoke(
    {
        "messages": messages + [HumanMessage(content="What icecream do I like?")],
        "language": "English"
    }
)
response.content

"I don't have any information about your preferences. If you want to tell me, I'd be happy to chat about your favorite ice cream flavors!"

In [ ]:
# Lets wrap this in the message history
with_message_history=RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="messages"
)
config = {
            "configurable":
            {"session_id": "chat5"}
        }

In [ ]:
response = with_message_history.invoke(
    {
        "messages": messages + [HumanMessage(content="Whats my name?")],
        "language": "English"
    },
    config=config
)

In [28]:
response.content

"I don't think we've introduced ourselves yet. I don't have any information about your name. Would you like to tell me?"